In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [ ]:
# 1. Initialize the Sequential model with an explicit input rank.
model = models.Sequential([
    layers.Input(shape=(8, 8), name='input_layer'),
    layers.Flatten(name='flatten'),
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(4, activation='softmax'),
])

# 6. Compile: Use Categorical Crossentropy for one-hot labels
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Show the summary (it will confirm the total params)
model.summary()

In [ ]:
def prepare_data(x):
    #med_filtered_frame = scipy.signal.medfilt(x, kernel_size=3)
    x_normalized = x / 255.0 if x.max() > 1.0 else x

    # 2. Apply threshold (0.5) and cast to integer
    x_binarized = (x_normalized > 0.5).astype(np.float32)
    return x_normalized

# Load the archive
test_data = np.load('arrows8_keras_format.npz')

# List the keys to see what's inside
print("Keys in file:", test_data.files)

# Extract images and labels (replace 'x_test'/'y_test' with your actual keys)
x_train = test_data['x_train']
y_train = test_data['y_train']

x_val = test_data['x_val']
y_val = test_data['y_val']

x_test = test_data['x_test']
y_test = test_data['y_test']

x_train = prepare_data(x_train)
x_val = prepare_data(x_val)
x_test = prepare_data(x_test)

print(f"Test data shape: {x_test.shape}")



In [ ]:
def display_gray_grid(x_test, y_test):
    # 'facecolor' sets the background color of the whole window
    fig = plt.figure(figsize=(7, 4)) 
    
    for i in range(15):
        # Set the background for each individual subplot
        ax = plt.subplot(3, 5, i + 1)
        
        img = np.squeeze(x_test[i])
        true_label = y_test.flatten()[i]
        named_label = ["up", "left", "down", "right"][true_label]
        
        # Display image
        plt.imshow(img, cmap='viridis', interpolation='nearest')
        
        # Change text color to white so it's visible on the gray background
        plt.title(f"{named_label}", fontsize=9)
        plt.axis('off')
    
    plt.subplots_adjust(wspace=0.1, hspace=0.4)
    plt.show()

# Call the function
display_gray_grid(x_val, y_val)

In [ ]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,       # reduce LR by 50%
    patience=50,       # wait 15 epochs with no improvement
    min_lr=1e-7
)

# If your y_train is one-hot (4274, 4), this works perfectly:
history = model.fit(x_train, y_train, 
                    epochs=20, 
                    batch_size=32, 
                    validation_data=(x_val, y_val), 
                    verbose=1,
                    callbacks = [])

In [ ]:
import matplotlib.pyplot as plt
def plot_history(history):
    # Plot Training & Validation Accuracy / Loss
    acc = history['accuracy']
    val_acc = history['val_accuracy']
    loss = history['loss']
    val_loss = history['val_loss']
    epochs = range(1, len(acc) + 1)
    # Plot Accuracy
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(epochs, acc, 'b-', label='Training Accuracy')
    plt.plot(epochs, val_acc, 'r--', label='Validation Accuracy')
    plt.title('Training & Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    # Plot Loss
    plt.subplot(1,2,2)
    plt.plot(epochs, loss, 'b-', label='Training Loss')
    plt.plot(epochs, val_loss, 'r--', label='Validation Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
plot_history(history.history)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
#Evaluate the model on test data
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

#Confusion Matrix
y_pred = np.argmax(model.predict(x_test), axis=1)
y_true = y_test.flatten()

cm = confusion_matrix(y_true, y_pred)
class_names = ['up', 'down', 'left', 'right']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
plt.figure(figsize=(10,10))
disp.plot(cmap='Blues', xticks_rotation=45, values_format='d')
plt.title("Confusion Matrix on Test Data")
plt.show()

In [ ]:
def display_pred_grid(x_test, y_test, predicted_labels):
    # 'facecolor' sets the background color of the whole window
    fig = plt.figure(figsize=(10, 6)) 
    
    for i in range(15):
        # Set the background for each individual subplot
        ax = plt.subplot(3, 5, i + 1)
        
        img = np.squeeze(x_test[i])
        true_label = y_test.flatten()[i]
        named_label = ["up", "left", "down", "right"][true_label]
        predicted_label = predicted_labels[i]
        named_predicted_label = ["up", "left", "down", "right"][predicted_label]
        
        # Display image
        plt.imshow(img, cmap='viridis', interpolation='nearest')
        
        # Change text color to white so it's visible on the gray background
        plt.title(f"True: {named_label}\nPredicted: {named_predicted_label}", fontsize=9)
        plt.axis('off')
    
    plt.subplots_adjust(wspace=0.1, hspace=0.4)
    plt.show()

# Call the function
display_pred_grid(x_test, y_test, y_pred)

In [ ]:
model.save('wedrowiec-pre_hls.h5')

## hls4ml conversion for hardware

These cells generate the working fixed-point HLS network directly from the trained notebook model. The default hardware model removes the final softmax and outputs four logits; use `argmax` on those logits to pick the arrow class.


In [ ]:
import shutil
from pathlib import Path

import hls4ml
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

saved_model_path = Path('wedrowiec-pre_hls.h5')
hls_project_name = 'wedrowiec_hls'
hls_output_dir = Path('hls4ml_projects') / hls_project_name
clean_hls_output = True

# 'logits' removes final softmax from HLS and outputs four fixed-point scores.
# Classify with np.argmax(scores, axis=1).
# 'one_hot' keeps a softmax-shaped output but implements it as hardware argmax.
hls_output_mode = 'logits'

try:
    keras_reference_model = model
except NameError:
    keras_reference_model = tf.keras.models.load_model(saved_model_path, compile=False)

def clone_without_final_softmax(keras_model):
    hls_ready_model = models.Sequential(name=f'{keras_model.name}_logits')
    hls_ready_model.add(layers.Input(shape=keras_model.input_shape[1:], name='input_layer'))

    copied_layers = []
    for layer in keras_model.layers:
        if isinstance(layer, layers.InputLayer):
            continue

        config = layer.get_config()
        if isinstance(layer, layers.Dense) and config.get('activation') == 'softmax':
            config['activation'] = 'linear'

        new_layer = layer.__class__.from_config(config)
        hls_ready_model.add(new_layer)
        copied_layers.append((layer, new_layer))

    for old_layer, new_layer in copied_layers:
        weights = old_layer.get_weights()
        if weights:
            new_layer.set_weights(weights)

    return hls_ready_model

if hls_output_mode == 'logits':
    keras_model_for_hls = clone_without_final_softmax(keras_reference_model)
    final_activation = getattr(keras_model_for_hls.layers[-1].activation, '__name__', '')
    if final_activation != 'linear':
        raise RuntimeError(f'Expected linear final HLS layer, got {final_activation}')
elif hls_output_mode == 'one_hot':
    keras_model_for_hls = keras_reference_model
else:
    raise ValueError("hls_output_mode must be 'logits' or 'one_hot'")

def match_model_input_shape(x, keras_model):
    x = np.asarray(x, dtype=np.float32)
    input_shape = tuple(keras_model.input_shape[1:])

    if any(dim is None for dim in input_shape):
        raise ValueError(f'Model input shape is not fully defined: {keras_model.input_shape}')

    if x.shape[1:] == input_shape:
        matched = x
    elif x.ndim == 4 and x.shape[-1] == 1 and x.shape[1:-1] == input_shape:
        matched = np.squeeze(x, axis=-1)
    elif len(input_shape) == 3 and input_shape[-1] == 1 and x.ndim == 3 and x.shape[1:] == input_shape[:-1]:
        matched = x[..., np.newaxis]
    else:
        matched = x.reshape((x.shape[0], *input_shape))

    matched = np.ascontiguousarray(matched, dtype=np.float32)
    print(f'Keras/HLS input shape: {matched.shape}, dtype: {matched.dtype}')
    return matched

print(f'HLS output mode: {hls_output_mode}')
print(f'HLS model final activation: {getattr(keras_model_for_hls.layers[-1].activation, "__name__", "n/a")}')


### Conversion configuration

In [ ]:
hls_backend = 'VivadoAccelerator'
board_name = 'pynq-z2'
fpga_part = 'xc7z020clg400-1'
clock_period = 20  # 50 MHz is a relaxed starting point for PYNQ-Z2 timing closure.
io_type = 'io_stream'
hls_precision = 'fixed<10,4,RND,SAT>'
hls_input_precision = 'ufixed<8,1,RND,SAT>'
hls_reuse_factor = 1

hls_config = hls4ml.utils.config_from_keras_model(
    keras_model_for_hls,
    granularity='name',
    backend=hls_backend,
    default_precision=hls_precision,
    default_reuse_factor=hls_reuse_factor,
)

hls_config['Model']['Strategy'] = 'Latency'
hls_config['LayerName']['input_layer']['Precision']['result'] = hls_input_precision

if hls_output_mode == 'logits' and 'dense_3_softmax' in hls_config['LayerName']:
    raise RuntimeError('Stale HLS model: dense_3_softmax is present. Rerun the HLS setup cell first.')

if hls_output_mode == 'one_hot' and 'dense_3_softmax' in hls_config['LayerName']:
    hls_config['LayerName']['dense_3_softmax']['Implementation'] = 'argmax'

hls_config


### Generate the hls4ml project

In [ ]:
if clean_hls_output and hls_output_dir.exists():
    shutil.rmtree(hls_output_dir)

hls_model = hls4ml.converters.convert_from_keras_model(
    keras_model_for_hls,
    hls_config=hls_config,
    output_dir=str(hls_output_dir),
    project_name=hls_project_name,
    backend=hls_backend,
    board=board_name,
    part=fpga_part,
    clock_period=clock_period,
    io_type=io_type,
    interface='axi_stream',
    driver='python',
)

hls_model.write()
firmware_cpp = hls_output_dir / 'firmware' / f'{hls_project_name}.cpp'
firmware_source = firmware_cpp.read_text()
if hls_output_mode == 'logits' and 'nnet::softmax' in firmware_source:
    raise RuntimeError('Generated stale softmax firmware. Restart the kernel and rerun the HLS cells from setup.')

print(f'hls4ml project written to: {hls_output_dir}')
print(f'Generated top function source: {firmware_cpp}')


## hls4ml model checks

Run these after the conversion cell. The first cell compiles the local hls4ml simulation model and compares class labels against the trained Keras model.


In [ ]:
if hls_output_mode == 'logits' and 'dense_3_softmax' in hls_config.get('LayerName', {}):
    raise RuntimeError('Stale HLS config has dense_3_softmax. Rerun the HLS setup/config/convert cells.')

hls_model.compile()

x_test_for_hls = match_model_input_shape(x_test, keras_model_for_hls)
y_true = y_test.flatten()

keras_input = tf.convert_to_tensor(x_test_for_hls, dtype=tf.float32)
reference_predictions = keras_reference_model(keras_input, training=False).numpy()
keras_hls_predictions = keras_model_for_hls(keras_input, training=False).numpy()
hls_predictions = hls_model.predict(x_test_for_hls)

reference_labels = np.argmax(reference_predictions, axis=1)
keras_hls_labels = np.argmax(keras_hls_predictions, axis=1)
hls_labels = np.argmax(hls_predictions, axis=1)

reference_accuracy = np.mean(reference_labels == y_true)
keras_hls_accuracy = np.mean(keras_hls_labels == y_true)
hls_accuracy = np.mean(hls_labels == y_true)
source_agreement = np.mean(keras_hls_labels == reference_labels)
hls_agreement = np.mean(hls_labels == reference_labels)

print(f'Keras reference test accuracy: {reference_accuracy * 100:.2f}%')
print(f'HLS-source Keras test accuracy: {keras_hls_accuracy * 100:.2f}%')
print(f'hls4ml test accuracy: {hls_accuracy * 100:.2f}%')
print(f'HLS-source/reference label agreement: {source_agreement * 100:.2f}%')
print(f'hls4ml/reference label agreement: {hls_agreement * 100:.2f}%')

if hls_output_mode == 'logits':
    max_abs_diff = np.max(np.abs(keras_hls_predictions - hls_predictions))
    mean_abs_diff = np.mean(np.abs(keras_hls_predictions - hls_predictions))
    print(f'Max abs logit difference: {max_abs_diff:.6f}')
    print(f'Mean abs logit difference: {mean_abs_diff:.6f}')

if hls_accuracy < 0.95:
    raise RuntimeError('hls4ml accuracy is too low; this is probably a stale softmax build or stale notebook kernel state.')


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

class_labels = ['up', 'left', 'down', 'right']

print(classification_report(y_true, hls_labels, target_names=class_labels))

cm = confusion_matrix(y_true, hls_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(cmap='Blues', values_format='d')
plt.title('hls4ml confusion matrix')
plt.show()

In [ ]:
display_pred_grid(x_test, y_test, hls_labels)

### Optional HLS build

In [ ]:
# Run this later when your Xilinx/Vivado or Vitis HLS toolchain is available.
# It can take a while and will create the HLS build outputs and reports.
# hls_model.build(csim=True, synth=True, cosim=False, export=True)
# hls4ml.report.read_vivado_report(str(hls_output_dir))